In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor, Pool
import warnings
warnings.filterwarnings('ignore')

# ========================
# 1. ĐỌC DỮ LIỆU
# ========================
df = pd.read_csv('vietnam_housing_dataset - Copy.csv')  # thay bằng đường dẫn file thực tế của bạn

print(f"Shape ban đầu: {df.shape}")

# Chuẩn hóa tên cột
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

# Chuyển đổi kiểu dữ liệu
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['area'] = pd.to_numeric(df['area'], errors='coerce')

df = df.dropna(subset=['price', 'area'])
df = df[(df['price'] > 0) & (df['area'] > 0)]

# ========================
# 2. CLIP OUTLIER (từ Code 1)
# ========================
lower_p, upper_p = df['price'].quantile([0.01, 0.99])
df['price'] = df['price'].clip(lower_p, upper_p)

lower_a, upper_a = df['area'].quantile([0.005, 0.995])
df['area'] = df['area'].clip(lower_a, upper_a)

print(f"Sau clip outlier: {df.shape[0]} dòng")

# ========================
# 3. EXTRACT CITY & DISTRICT (hàm tinh xảo từ Code 1)
# ========================
def extract_district_city_v2(address):
    if pd.isna(address) or not isinstance(address, str):
        return 'Other', 'Other'
    
    address_lower = address.lower().strip()
    
    # City mapping
    city_map = {
        'hồ chí minh': 'HCM', 'tp.hồ chí minh': 'HCM', 'tphcm': 'HCM', 'hcm': 'HCM',
        'hà nội': 'HN', 'thành phố hà nội': 'HN', 'hn': 'HN',
        'hưng yên': 'HY', 'bình dương': 'BD', 'đà nẵng': 'ĐN', 'đn': 'ĐN',
        'long an': 'LA', 'đồng nai': 'Đồng Nai', 'bà rịa vũng tàu': 'BRVT',
        'quảng ninh': 'QN', 'phú thọ': 'PT', 'hải phòng': 'HP', 'khánh hòa': 'KH',
        'kiên giang': 'KG', 'bình thuận': 'BT', 'hải dương': 'HD', 'thanh hóa': 'TH',
    }
    city = 'Other'
    for k, v in city_map.items():
        if k in address_lower:
            city = v
            break
    
    # Tìm district/quận/huyện bằng từ khóa
    district_keywords = ['quận ', 'huyện ', 'thị trấn ', 'phường ', 'xã ', 'thị xã ', 'tp.', 'thành phố ']
    district = 'Other'
    
    # Tách theo dấu phẩy
    parts = [p.strip() for p in address.split(',')]
    
    for part in parts:
        part_lower = part.lower()
        # Nếu phần này chứa từ khóa quận/huyện/phường/xã → coi là district
        if any(kw in part_lower for kw in district_keywords):
            # Lấy tên sạch hơn
            for kw in district_keywords:
                part = part.replace(kw, '').strip()
            district = part.title()
            break
        # Nếu không có từ khóa nhưng là tên quận/huyện quen thuộc
        elif any(d in part_lower for d in ['gò vấp', 'bình thạnh', 'quận 7', 'thủ đức', 'long biên', 'cầu giấy', 'đống đa', 'thanh xuân']):
            district = part.title()
            break
    
    # Fix tên quận/huyện phổ biến
    district_fix = {
        'gò vấp': 'Gò Vấp', 'bình thạnh': 'Bình Thạnh', 'quận 7': 'Quận 7',
        'thủ đức': 'Thủ Đức', 'long biên': 'Long Biên', 'cầu giấy': 'Cầu Giấy',
        'đống đa': 'Đống Đa', 'thanh xuân': 'Thanh Xuân', 'hà đông': 'Hà Đông',
        'nam từ liêm': 'Nam Từ Liêm', 'tây hồ': 'Tây Hồ', 'hoàn kiếm': 'Hoàn Kiếm',
        'bắc từ liêm': 'Bắc Từ Liêm', 'hoàng mai': 'Hoàng Mai', 'hai bà trưng': 'Hai Bà Trưng',
    }
    district_lower = district.lower()
    for k, v in district_fix.items():
        if k in district_lower:
            district = v
            break
    
    if district == 'Other' and 'dự án' in address_lower:
        # Với dự án lớn, thường có tên quận/huyện trong tên dự án
        if 'ocean park' in address_lower:
            district = 'Văn Giang'
        elif 'grand park' in address_lower:
            district = 'Quận 9'
        elif 'vinhomes' in address_lower and 'ocean' in address_lower:
            district = 'Văn Giang'
        # thêm rule nếu bạn thấy nhiều dự án cụ thể
    
    return city, district

df['city'], df['district'] = zip(*df['address'].apply(extract_district_city_v2))
df = df.drop('address', axis=1)  # Drop address sau extract như Code 2

print("\nPhân bố City sau extract:")
print(df['city'].value_counts().head(15))
print("\nPhân bố District sau extract (top 20):")
print(df['district'].value_counts().head(20))

# Tạo log_price trước split
df['log_price'] = np.log1p(df['price'])

# ========================
# 4. TRAIN_TEST_SPLIT ĐẦU TIÊN ĐỂ TRÁNH LEAKAGE
# ========================
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)

print(f"\nTrain: {len(train_df):,} | Test: {len(test_df):,}")

# ========================
# 5. XỬ LÝ MISSING VALUES (groupby district từ Code 2, trên train rồi apply test)
# ========================
numerical_cols = ['area', 'frontage', 'access_road', 'floors', 'bedrooms', 'bathrooms']

# Tính median theo district trên train
district_medians = {}
for col in numerical_cols:
    if col in train_df.columns:
        district_medians[col] = train_df.groupby('district')[col].median().to_dict()
        overall_median = train_df[col].median()
        
        # Fill train
        train_df[col] = train_df.apply(
            lambda row: district_medians[col].get(row['district'], overall_median) if pd.isna(row[col]) else row[col],
            axis=1
        )
        # Fill remaining with overall
        train_df[col] = train_df[col].fillna(overall_median)
        
        # Fill test similarly using train's medians
        test_df[col] = test_df.apply(
            lambda row: district_medians[col].get(row['district'], overall_median) if pd.isna(row[col]) else row[col],
            axis=1
        )
        test_df[col] = test_df[col].fillna(overall_median)

categorical_cols = ['house_direction', 'balcony_direction', 'legal_status', 'furniture_state']

for col in categorical_cols:
    if col in train_df.columns:
        train_df[col] = train_df[col].fillna('Missing')
        test_df[col] = test_df[col].fillna('Missing')

# Drop balcony_direction nếu missing quá nhiều trên train (từ Code 2)
if 'balcony_direction' in train_df.columns:
    missing_rate = (train_df['balcony_direction'] == 'Missing').mean()
    if missing_rate > 0.7:
        print(f"Drop balcony_direction vì missing rate: {missing_rate:.2%}")
        train_df = train_df.drop('balcony_direction', axis=1)
        test_df = test_df.drop('balcony_direction', axis=1)
        if 'balcony_direction' in categorical_cols:
            categorical_cols.remove('balcony_direction')

# ========================
# 6. FEATURE ENGINEERING (từ Code 1, nhưng tính avg trên train rồi map)
# ========================
# Tính price_per_m2 trên train và test (sử dụng price thực, nhưng avg chỉ trên train)
train_df['price_per_m2'] = train_df['price'] / train_df['area']
test_df['price_per_m2'] = test_df['price'] / test_df['area']

# District avg price/m² chỉ trên train
district_avg = train_df.groupby('district')['price_per_m2'].mean().to_dict()
overall_avg = train_df['price_per_m2'].mean()

train_df['district_avg_price_per_m2'] = train_df['district'].map(district_avg).fillna(overall_avg)
test_df['district_avg_price_per_m2'] = test_df['district'].map(district_avg).fillna(overall_avg)

# Các feature khác
for d in [train_df, test_df]:
    d['total_rooms'] = d['bedrooms'].fillna(0) + d['bathrooms'].fillna(0) + 1
    d['bed_bath_ratio'] = d['bedrooms'] / (d['bathrooms'] + 1e-6)
    d['area_per_floor'] = d['area'] / (d['floors'] + 1e-6)
    d['rooms_per_floor'] = d['total_rooms'] / (d['floors'] + 1e-6)

    good_dirs = ['Nam', 'Đông - Nam', 'Đông', 'Đông Nam']
    d['is_good_direction'] = d['house_direction'].isin(good_dirs).astype(int)

    # Feature interaction
    d['area_x_district_avg'] = d['area'] * d['district_avg_price_per_m2']
    d['frontage_x_access'] = d['frontage'] * d['access_road']

# Drop price_per_m2 sau khi dùng (không cần cho model)
train_df = train_df.drop('price_per_m2', axis=1)
test_df = test_df.drop('price_per_m2', axis=1)

# Categorical as category
cat_cols = ['house_direction', 'balcony_direction', 'legal_status', 
            'furniture_state', 'city', 'district']

for col in cat_cols:
    if col in train_df.columns:
        train_df[col] = train_df[col].astype('category')
        test_df[col] = test_df[col].astype('category')

# ========================
# 7. CHUẨN BỊ FEATURES
# ========================
features = [
    'area', 'frontage', 'access_road', 'floors', 'bedrooms', 'bathrooms',
    'total_rooms', 'bed_bath_ratio', 'area_per_floor', 'rooms_per_floor',
    'is_good_direction', 'district_avg_price_per_m2',
    'area_x_district_avg', 'frontage_x_access',
    'house_direction', 'balcony_direction', 'legal_status', 'furniture_state',
    'city', 'district'
]

# Lọc features có tồn tại
features = [f for f in features if f in train_df.columns]

X_train = train_df[features].copy()
y_train = train_df['log_price']

X_test = test_df[features].copy()
y_test = test_df['log_price']

# ========================
# 8. TRAIN CATBOOST
# ========================
cat_features = [f for f in features if f in cat_cols]

train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool  = Pool(X_test, y_test, cat_features=cat_features)

model = CatBoostRegressor(
    iterations=7000,
    learning_rate=0.012,
    depth=8,
    l2_leaf_reg=6,
    random_seed=42,
    loss_function='RMSE',
    eval_metric='MAE',
    early_stopping_rounds=180,
    verbose=200,
    use_best_model=True
)

model.fit(
    train_pool,
    eval_set=test_pool,
)

# ========================
# 9. ĐÁNH GIÁ (kết hợp từ Code 2)
# ========================
y_pred_log = model.predict(X_test)

print("\n=== Log Price Metrics ===")
print(f"MAE: {mean_absolute_error(y_test, y_pred_log):.5f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_log)):.5f}")
print(f"R²: {r2_score(y_test, y_pred_log):.5f}")

y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log)

print("\n=== Giá thật (tỷ VND) ===")
print(f"MAE: {mean_absolute_error(y_test_real, y_pred_real):,.2f} tỷ")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_real, y_pred_real)):.2f} tỷ")
print(f"R²: {r2_score(y_test_real, y_pred_real):.4f}")

# Feature importance
fi = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=False).head(15)

print("\n=== Top 15 features quan trọng nhất ===")
print(fi.to_string(index=False))

Shape ban đầu: (30229, 12)
Sau clip outlier: 30229 dòng

Phân bố City sau extract:
city
HCM         11785
HN          10459
BD           1675
ĐN           1450
Other        1047
Đồng Nai      844
HP            776
KH            725
HY            404
LA            341
BRVT          240
BT            127
QN            111
TH            110
KG             83
Name: count, dtype: int64

Phân bố District sau extract (top 20):
district
Phường 5                  449
Phường 11                 439
Phường 12                 439
Phường 14                 385
Phường 13                 340
Phường 15                 336
Phường 3                  330
Phường 10                 321
Xã Long Hưng              287
Phường 4                  275
Phường 7                  247
Phường 8                  237
Phường 6                  231
Phường 9                  231
Phường Tân Quý            222
Phường 1                  218
Phường Tân Đông Hiệp      215
Phường Phú Hữu            210
Phường Thạch Bàn          2

In [2]:

# ========================
# 9. ĐÁNH GIÁ (kết hợp từ Code 2)
# ========================
y_pred_log = model.predict(X_test)

print("\n=== Log Price Metrics ===")
print(f"MAE: {mean_absolute_error(y_test, y_pred_log):.5f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_log)):.5f}")
print(f"R²: {r2_score(y_test, y_pred_log):.5f}")

y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log)

print("\n=== Giá thật (tỷ VND) ===")
print(f"MAE: {mean_absolute_error(y_test_real, y_pred_real):,.2f} tỷ")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_real, y_pred_real)):.2f} tỷ")
print(f"R²: {r2_score(y_test_real, y_pred_real):.4f}")

# Feature importance
fi = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=False).head(15)

print("\n=== Top 15 features quan trọng nhất ===")
print(fi.to_string(index=False))


=== Log Price Metrics ===
MAE: 0.14303
RMSE: 0.20436
R²: 0.67407

=== Giá thật (tỷ VND) ===
MAE: 0.95 tỷ
RMSE: 1.32 tỷ
R²: 0.6401

=== Top 15 features quan trọng nhất ===
                  feature  importance
      area_x_district_avg   35.736633
                     city    8.615296
                 district    7.294396
              access_road    6.336848
                   floors    5.140668
district_avg_price_per_m2    5.019091
        frontage_x_access    4.678302
              total_rooms    3.682870
           area_per_floor    3.265391
                     area    3.252022
                bathrooms    3.012034
          rooms_per_floor    2.533525
             legal_status    2.467368
                 frontage    2.323113
          furniture_state    2.113032


In [7]:
import pickle
# Lưu các biến cần cho inference
preprocess_globals = {
    'district_medians': district_medians,
    'district_avg': district_avg,
    'overall_avg': overall_avg,
    'overall_medians': {col: train_df[col].median() for col in numerical_cols if col in train_df.columns},
    'features': features,
    'cat_cols': cat_cols,
    'good_dirs': good_dirs,
    'numerical_cols': numerical_cols,  # để biết các cột cần fill
}

with open('preprocess_globals.pkl', 'wb') as f:
    pickle.dump(preprocess_globals, f)

print("Đã lưu preprocess_globals.pkl")
print("Bây giờ bạn có thể dùng 2 file này trong predict_price.py")

Đã lưu preprocess_globals.pkl
Bây giờ bạn có thể dùng 2 file này trong predict_price.py


In [ ]:
import pandas as pd
import numpy as np

# Tạo dữ liệu test mẫu
data_temp = {
    'price': [4500, 10500, 3200, 7800, 15000, 6200, 8900, 2800, 13500, 5500],
    'area': [65, 120, 48, 95, 180, 75, 110, 42, 150, 80],
    'address': [
        'Quận 7, TP. Hồ Chí Minh',
        'Thủ Đức, Thành phố Hồ Chí Minh',
        'Cầu Giấy, Hà Nội',
        'Bình Thạnh, HCM',
        'Quận 2, TP.HCM',
        'Long Biên, Hà Nội',
        'Quận 9, TP. Hồ Chí Minh',
        'Hoàn Kiếm, Hà Nội',
        'Nam Từ Liêm, Hà Nội',
        'Gò Vấp, HCM'
    ],
    'frontage': [5.2, 8.0, 4.0, 6.5, 10.0, 5.8, 7.2, 3.8, 9.0, 5.5],
    'access_road': [6.0, 12.0, 4.5, 8.0, 15.0, 7.0, 10.0, 3.5, 11.0, 6.5],
    'floors': [4, 6, 3, 5, 8, 4, 7, 2, 9, 5],
    'bedrooms': [3, 4, 2, 3, 5, 3, 4, 2, 4, 3],
    'bathrooms': [3, 4, 2, 3, 5, 3, 4, 2, 4, 2],
    'house_direction': ['Nam', 'Đông Nam', 'Tây', 'Đông', 'Nam', 'Bắc', 'Đông Nam', 'Tây Bắc', 'Nam', 'Đông'],
    'balcony_direction': ['Nam', 'Đông Nam', np.nan, 'Đông', 'Nam', np.nan, 'Đông Nam', 'Tây', 'Nam', np.nan],
    'legal_status': ['Sổ hồng riêng', 'Sổ hồng riêng', 'Đang chờ sổ', 'Sổ hồng riêng', 'Sổ hồng riêng', 'Sổ đỏ', 'Sổ hồng riêng', 'Đang chờ sổ', 'Sổ hồng riêng', 'Sổ đỏ'],
    'furniture_state': ['Nội thất đầy đủ', 'Nội thất cao cấp', 'Không nội thất', 'Nội thất cơ bản', 'Nội thất đầy đủ', 'Nội thất cơ bản', 'Nội thất cao cấp', 'Không nội thất', 'Nội thất đầy đủ', 'Nội thất cơ bản']
}

df_temp = pd.DataFrame(data_temp)

print("Dữ liệu mẫu (10 nhà):")
print(df_temp)
print("\nShape:", df_temp.shape)

In [ ]:
# ========================
# Tiền xử lý df_temp giống hệt pipeline cũ
# ========================

# Chuẩn hóa tên cột
df_temp.columns = df_temp.columns.str.strip().str.replace(' ', '_').str.lower()

# Chuyển numeric
numeric_cols = ['price', 'area', 'frontage', 'access_road', 'floors', 'bedrooms', 'bathrooms']
for col in numeric_cols:
    if col in df_temp.columns:
        df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')

df_temp = df_temp.dropna(subset=['area'])  # ít nhất phải có area
df_temp = df_temp[df_temp['area'] > 0]

# Clip outlier (tạm dùng quantile của temp, hoặc bỏ nếu muốn giữ nguyên)
lower_a, upper_a = df_temp['area'].quantile([0.005, 0.995])
df_temp['area'] = df_temp['area'].clip(lower_a, upper_a)

# Extract city & district (phải có hàm extract_district_city_v2 đã định nghĩa)
df_temp['city'], df_temp['district'] = zip(*df_temp['address'].apply(extract_district_city_v2))
df_temp = df_temp.drop('address', axis=1)

# Fill missing numerical - dùng medians từ TRAIN
numerical_cols_to_fill = ['area', 'frontage', 'access_road', 'floors', 'bedrooms', 'bathrooms']

for col in numerical_cols_to_fill:
    if col in df_temp.columns:
        # Fill theo district median từ train
        df_temp[col] = df_temp.apply(
            lambda row: district_medians.get(col, {}).get(row['district'], np.nan)
            if pd.isna(row[col]) else row[col],
            axis=1
        )
        
        # Fallback median: ưu tiên từ train nếu còn train_df, không thì từ temp
        if 'train_df' in globals() and col in train_df.columns:
            fallback = train_df[col].median()
        else:
            fallback = df_temp[col].median()
            print(f"Warning: dùng median của df_temp cho '{col}' vì không tìm thấy train_df")
        
        df_temp[col] = df_temp[col].fillna(fallback)

# Fill categorical
categorical_cols = ['house_direction', 'balcony_direction', 'legal_status', 'furniture_state']
for col in categorical_cols:
    if col in df_temp.columns:
        df_temp[col] = df_temp[col].fillna('Missing')

# Nếu missing balcony_direction quá nhiều → drop (tùy model của bạn)
if 'balcony_direction' in df_temp.columns:
    missing_rate = (df_temp['balcony_direction'] == 'Missing').mean()
    if missing_rate > 0.7:
        df_temp = df_temp.drop('balcony_direction', axis=1)

# Feature engineering (giống hệt lúc train)
df_temp['price_per_m2'] = df_temp['price'] / df_temp['area']   # tạm tính để tạo feature

df_temp['district_avg_price_per_m2'] = df_temp['district'].map(district_avg).fillna(overall_avg)

for d in [df_temp]:
    d['total_rooms'] = d['bedrooms'].fillna(0) + d['bathrooms'].fillna(0) + 1
    d['bed_bath_ratio'] = d['bedrooms'] / (d['bathrooms'] + 1e-6)
    d['area_per_floor'] = d['area'] / (d['floors'] + 1e-6)
    d['rooms_per_floor'] = d['total_rooms'] / (d['floors'] + 1e-6)
    good_dirs = ['Nam', 'Đông - Nam', 'Đông', 'Đông Nam']  # tùy bạn chỉnh
    d['is_good_direction'] = d['house_direction'].isin(good_dirs).astype(int)
    d['area_x_district_avg'] = d['area'] * d['district_avg_price_per_m2']
    d['frontage_x_access'] = d['frontage'] * d['access_road']

df_temp = df_temp.drop('price_per_m2', axis=1, errors='ignore')

# Chọn features giống lúc train
X_temp = df_temp[features].copy()

# Đảm bảo category
for col in cat_cols:
    if col in X_temp.columns:
        X_temp[col] = X_temp[col].astype('category')

# ========================
# Dự đoán
# ========================
y_pred_log_temp = model.predict(X_temp)
y_pred_real_temp = np.expm1(y_pred_log_temp)

# Kết quả
# Sau khi predict y_pred_real_temp = np.expm1(y_pred_log_temp)

result = df_temp[['area', 'district', 'city', 'bedrooms', 'floors', 'price']].copy()
result['predicted_price'] = y_pred_real_temp.round(2)          # giờ là tỷ VND
result['diff_tỷ'] = (result['predicted_price'] - result['price']).round(2)
result['diff_%'] = ((result['predicted_price'] / result['price'] - 1) * 100).round(1)

print("\nKết quả dự đoán trên dữ liệu mẫu (đơn vị: tỷ VND):")
print(result[['area', 'district', 'city', 'bedrooms', 'floors', 'price', 'predicted_price', 'diff_tỷ', 'diff_%']].to_string(index=False))